# Lab 5 — Machine Learning with Scikit-learn and From Scratch

## DS605 — Fundamentals of Machine Learning

This notebook implements the complete assignment workflow on the **UCI Productivity Prediction of Garment Employees** dataset.

### Tasks
1. **Part A — Scikit-learn**
   - Preprocess the data.
   - Train Linear Regression for `actual_productivity`.
   - Train Logistic Regression for `MeetsTarget`.
   - Evaluate performance and execution time.

2. **Part B — NumPy/Pandas from scratch**
   - Rebuild preprocessing.
   - Implement Linear Regression using the pseudo-inverse.
   - Implement Logistic Regression using the sigmoid function and gradient descent.
   - Implement evaluation metrics manually.

3. **Part C — Comparison and optimization**
   - Compare predictive performance and execution time.
   - Examine the effect of a different learning-rate/epoch configuration for manual Logistic Regression.
   - Explain the implementation differences.

## 1. Dataset and Problem Definition

The dataset contains information about garment-production teams and their productivity.

Two machine-learning problems are created:

### Regression
Predict the continuous target:

`actual_productivity`

using **Linear Regression**.

### Classification
Create a binary target:

`MeetsTarget = 1` when `actual_productivity >= targeted_productivity`, otherwise `0`.

Then predict `MeetsTarget` using **Logistic Regression**.

`actual_productivity` is not used as an input feature for classification because it is the quantity from which the classification target is derived.

In [1]:
import time
import numpy as np
import pandas as pd

from ucimlrepo import fetch_ucirepo

# Fetch dataset
dataset = fetch_ucirepo(id=597)

X = dataset.data.features.copy()
y = dataset.data.targets.copy()

print("Feature shape:", X.shape)
print("Target shape :", y.shape)
X.head()

Feature shape: (1197, 14)
Target shape : (1197, 1)


,date,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers
0,1/1/2015,Quarter1,sweing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0
1,1/1/2015,Quarter1,finishing,Thursday,1,0.75,3.94,NaN,960,0,0.0,0,0,8.0
2,1/1/2015,Quarter1,sweing,Thursday,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5
3,1/1/2015,Quarter1,sweing,Thursday,12,0.80,11.41,968.0,3660,50,0.0,0,0,30.5
4,1/1/2015,Quarter1,sweing,Thursday,6,0.80,25.90,1170.0,1920,50,0.0,0,0,56.0


In [2]:
print("Feature columns:")
print(X.columns.tolist())

print("\nData types:")
print(X.dtypes)

print("\nMissing values:")
print(X.isnull().sum())

print("\nRegression target:")
print(y["actual_productivity"].describe())

Feature columns:
['date', 'quarter', 'department', 'day', 'team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers']

Data types:
date                         str
quarter                      str
department                   str
day                          str
team                       int64
targeted_productivity    float64
smv                      float64
wip                      float64
over_time                  int64
incentive                  int64
idle_time                float64
idle_men                   int64
no_of_style_change         int64
no_of_workers            float64
dtype: object

Missing values:
date                       0
quarter                    0
department                 0
day                        0
team                       0
targeted_productivity      0
smv                        0
wip                      506
over_time                  0
incentive                  0
idle_time

## 2. Create the Classification Target

The assignment defines `MeetsTarget` from the relationship between actual and targeted productivity.

The original regression target is kept separately as `y_reg`, while the classification target is kept as `y_clf`.

In [3]:
y["MeetsTarget"] = (
    y["actual_productivity"] >= X["targeted_productivity"]
).astype(int)

y_reg = y["actual_productivity"].copy()
y_clf = y["MeetsTarget"].copy()

print(y_clf.value_counts())

MeetsTarget
1    875
0    322
Name: count, dtype: int64


## 3. Feature Groups

For this implementation:

- `date`, `quarter`, `department`, and `day` are treated as categorical features.
- The remaining predictor columns are treated as numerical features.
- `wip` contains missing values. Its distribution is right-skewed, so the **median** is used for imputation.
- Preprocessing statistics are learned from the training set only and then applied to the test set.

In [4]:
categorical_cols = [
    "date",
    "quarter",
    "department",
    "day"
]

numerical_cols = [
    "team",
    "targeted_productivity",
    "smv",
    "wip",
    "over_time",
    "incentive",
    "idle_time",
    "idle_men",
    "no_of_style_change",
    "no_of_workers"
]

print("Categorical features:", categorical_cols)
print("Numerical features  :", numerical_cols)

Categorical features: ['date', 'quarter', 'department', 'day']
Numerical features  : ['team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers']


## 4. Fixed Train-Test Split

A single fixed split is created and reused for **both regression and classification** so that the two implementations are compared fairly.

The split is created once here. The manual implementation below does not call any scikit-learn preprocessing, model, metric, or train-test utility.

In [5]:
from sklearn.model_selection import train_test_split

indices = np.arange(len(X))

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=42
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_reg_train = y_reg.iloc[train_idx].copy()
y_reg_test = y_reg.iloc[test_idx].copy()

y_clf_train = y_clf.iloc[train_idx].copy()
y_clf_test = y_clf.iloc[test_idx].copy()

print("Training samples:", len(train_idx))
print("Testing samples :", len(test_idx))

Training samples: 957
Testing samples : 240


# Part A — Scikit-learn Implementation

## 5. Scikit-learn Preprocessing

### Numerical features
1. Missing values are filled using the training-set median.
2. Numerical features are standardized.

### Categorical features
1. Missing values are filled using the training-set mode.
2. Categories are one-hot encoded.

`fit_transform()` is used only on the training data. The test data is transformed using the already-fitted preprocessing object, preventing data leakage.

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape :", X_test_processed.shape)

Processed training shape: (957, 82)
Processed testing shape : (240, 82)


## 6. Scikit-learn Linear Regression

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

linear_model = LinearRegression()

start = time.perf_counter()
linear_model.fit(X_train_processed, y_reg_train)
linear_train_time = time.perf_counter() - start

start = time.perf_counter()
y_reg_pred = linear_model.predict(X_test_processed)
linear_pred_time = time.perf_counter() - start

linear_mae = mean_absolute_error(y_reg_test, y_reg_pred)
linear_rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))
linear_r2 = r2_score(y_reg_test, y_reg_pred)

print("Scikit-learn Linear Regression")
print("MAE             :", linear_mae)
print("RMSE            :", linear_rmse)
print("R²              :", linear_r2)
print("Training time   :", linear_train_time)
print("Prediction time :", linear_pred_time)

Scikit-learn Linear Regression
MAE             : 0.11238055624401078
RMSE            : 0.150047508664101
R²              : 0.15208423909412072
Training time   : 0.010270099999615923
Prediction time : 0.0005071000050520524


## 7. Scikit-learn Logistic Regression

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

logistic_model = LogisticRegression(max_iter=1000)

start = time.perf_counter()
logistic_model.fit(X_train_processed, y_clf_train)
logistic_train_time = time.perf_counter() - start

start = time.perf_counter()
y_clf_pred = logistic_model.predict(X_test_processed)
logistic_pred_time = time.perf_counter() - start

logistic_accuracy = accuracy_score(y_clf_test, y_clf_pred)
logistic_precision = precision_score(y_clf_test, y_clf_pred)
logistic_recall = recall_score(y_clf_test, y_clf_pred)
logistic_f1 = f1_score(y_clf_test, y_clf_pred)

print("Scikit-learn Logistic Regression")
print("Accuracy        :", logistic_accuracy)
print("Precision       :", logistic_precision)
print("Recall          :", logistic_recall)
print("F1              :", logistic_f1)
print("Training time   :", logistic_train_time)
print("Prediction time :", logistic_pred_time)

Scikit-learn Logistic Regression
Accuracy        : 0.7416666666666667
Precision       : 0.7699530516431925
Recall          : 0.9265536723163842
F1              : 0.841025641025641
Training time   : 0.033639499997661915
Prediction time : 0.0006414999952539802


# Part B — NumPy/Pandas Implementation

## 8. Manual Preprocessing

The preprocessing is rebuilt using only **Pandas and NumPy**.

The same training/test split and feature groups are used as in Part A.

For numerical features:
- missing values are replaced by training-set medians;
- training-set means and standard deviations are used for standardization.

For categorical features:
- missing values, if present, are replaced by the training-set mode;
- Pandas `get_dummies()` performs one-hot encoding;
- test columns are reindexed to exactly match the training columns.

In [9]:
X_train_manual = X_train.copy()
X_test_manual = X_test.copy()

# ----- Missing-value handling -----

# Numerical columns: training-set median
train_medians = X_train_manual[numerical_cols].median()

X_train_manual[numerical_cols] = (
    X_train_manual[numerical_cols].fillna(train_medians)
)
X_test_manual[numerical_cols] = (
    X_test_manual[numerical_cols].fillna(train_medians)
)

# Categorical columns: training-set mode
for col in categorical_cols:
    mode = X_train_manual[col].mode()
    if len(mode) > 0:
        X_train_manual[col] = X_train_manual[col].fillna(mode.iloc[0])
        X_test_manual[col] = X_test_manual[col].fillna(mode.iloc[0])

# ----- Standardization -----
# ddof=0 matches StandardScaler's population standard deviation.
train_means = X_train_manual[numerical_cols].mean()
train_stds = X_train_manual[numerical_cols].std(ddof=0)

X_train_manual[numerical_cols] = (
    X_train_manual[numerical_cols] - train_means
) / train_stds

X_test_manual[numerical_cols] = (
    X_test_manual[numerical_cols] - train_means
) / train_stds

# ----- One-hot encoding -----
X_train_cat = pd.get_dummies(
    X_train_manual[categorical_cols],
    dtype=float
)

X_test_cat = pd.get_dummies(
    X_test_manual[categorical_cols],
    dtype=float
)

# Make test columns identical to training columns
X_test_cat = X_test_cat.reindex(
    columns=X_train_cat.columns,
    fill_value=0
)

# Combine numerical and categorical features
X_train_manual_final = pd.concat(
    [X_train_manual[numerical_cols], X_train_cat],
    axis=1
)

X_test_manual_final = pd.concat(
    [X_test_manual[numerical_cols], X_test_cat],
    axis=1
)

X_train_np = X_train_manual_final.to_numpy(dtype=float)
X_test_np = X_test_manual_final.to_numpy(dtype=float)

print("Manual training shape:", X_train_np.shape)
print("Manual testing shape :", X_test_np.shape)

Manual training shape: (957, 82)
Manual testing shape : (240, 82)


## 9. Manual Regression Metrics

The manual section does not use scikit-learn metrics.

The metrics are calculated directly with NumPy:

- **MAE** — mean absolute error
- **RMSE** — square root of mean squared error
- **R²** — proportion of target variance explained by the model

In [10]:
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))


def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


def r2_score_manual(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)

## 10. Manual Linear Regression

For linear regression, the parameters are obtained using the pseudo-inverse:

`theta = pinv(X) @ y`

An intercept column of ones is added to the feature matrix.

In [11]:
X_train_lr = np.c_[
    np.ones(X_train_np.shape[0]),
    X_train_np
]

X_test_lr = np.c_[
    np.ones(X_test_np.shape[0]),
    X_test_np
]

start = time.perf_counter()

theta_linear = np.linalg.pinv(X_train_lr) @ y_reg_train.to_numpy()

linear_manual_train_time = time.perf_counter() - start

start = time.perf_counter()

y_reg_pred_manual = X_test_lr @ theta_linear

linear_manual_pred_time = time.perf_counter() - start

linear_manual_mae = mae(
    y_reg_test.to_numpy(),
    y_reg_pred_manual
)

linear_manual_rmse = rmse(
    y_reg_test.to_numpy(),
    y_reg_pred_manual
)

linear_manual_r2 = r2_score_manual(
    y_reg_test.to_numpy(),
    y_reg_pred_manual
)

print("Manual Linear Regression")
print("MAE             :", linear_manual_mae)
print("RMSE            :", linear_manual_rmse)
print("R²              :", linear_manual_r2)
print("Training time   :", linear_manual_train_time)
print("Prediction time :", linear_manual_pred_time)

Manual Linear Regression
MAE             : 0.11238111278338532
RMSE            : 0.1500482193646943
R²              : 0.15207620676266964
Training time   : 0.018172299998695962
Prediction time : 0.000112900001113303


## 11. Manual Logistic Regression

The model follows the standard logistic-regression workflow:

1. Compute the linear combination `z = X @ theta`.
2. Convert it to a probability using the sigmoid function.
3. Compute the gradient of the logistic loss.
4. Update the parameters using gradient descent.
5. Convert probabilities to classes using a threshold of `0.5`.

All matrix operations are vectorized with NumPy.

In [12]:
X_train_log = np.c_[
    np.ones(X_train_np.shape[0]),
    X_train_np
]

X_test_log = np.c_[
    np.ones(X_test_np.shape[0]),
    X_test_np
]

y_train_log = y_clf_train.to_numpy()
y_test_log = y_clf_test.to_numpy()


def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))


def logistic_predict_proba(X, theta):
    return sigmoid(X @ theta)


def logistic_predict(X, theta, threshold=0.5):
    probabilities = logistic_predict_proba(X, theta)
    return (probabilities >= threshold).astype(int)

In [13]:
learning_rate = 0.01
epochs = 10000

theta_log = np.zeros(X_train_log.shape[1])
n = X_train_log.shape[0]

start = time.perf_counter()

for _ in range(epochs):
    z = X_train_log @ theta_log
    y_prob = sigmoid(z)

    gradient = (
        X_train_log.T @ (y_prob - y_train_log)
    ) / n

    theta_log -= learning_rate * gradient

logistic_manual_train_time = time.perf_counter() - start

start = time.perf_counter()

y_test_prob = logistic_predict_proba(
    X_test_log,
    theta_log
)

y_clf_pred_manual = (
    y_test_prob >= 0.5
).astype(int)

logistic_manual_pred_time = time.perf_counter() - start

In [14]:
def classification_metrics(y_true, y_pred):
    accuracy = np.mean(y_true == y_pred)

    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    precision = (
        tp / (tp + fp)
        if (tp + fp) != 0
        else 0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) != 0
        else 0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) != 0
        else 0
    )

    return accuracy, precision, recall, f1


(
    manual_accuracy,
    manual_precision,
    manual_recall,
    manual_f1
) = classification_metrics(
    y_test_log,
    y_clf_pred_manual
)

print("Manual Logistic Regression")
print("Accuracy        :", manual_accuracy)
print("Precision       :", manual_precision)
print("Recall          :", manual_recall)
print("F1              :", manual_f1)
print("Training time   :", logistic_manual_train_time)
print("Prediction time :", logistic_manual_pred_time)

Manual Logistic Regression
Accuracy        : 0.7541666666666667
Precision       : 0.7681818181818182
Recall          : 0.9548022598870056
F1              : 0.8513853904282117
Training time   : 0.8485885999980383
Prediction time : 0.00022130000434117392


# Part C — Comparison and Optimization

## 12. Model Comparison

The regression and classification metrics are shown separately because they measure different tasks.

In [15]:
regression_comparison = pd.DataFrame({
    "Model": [
        "Scikit-learn Linear Regression",
        "Manual Linear Regression"
    ],
    "MAE": [
        linear_mae,
        linear_manual_mae
    ],
    "RMSE": [
        linear_rmse,
        linear_manual_rmse
    ],
    "R²": [
        linear_r2,
        linear_manual_r2
    ],
    "Training Time (s)": [
        linear_train_time,
        linear_manual_train_time
    ],
    "Prediction Time (s)": [
        linear_pred_time,
        linear_manual_pred_time
    ]
})

classification_comparison = pd.DataFrame({
    "Model": [
        "Scikit-learn Logistic Regression",
        "Manual Logistic Regression"
    ],
    "Accuracy": [
        logistic_accuracy,
        manual_accuracy
    ],
    "Precision": [
        logistic_precision,
        manual_precision
    ],
    "Recall": [
        logistic_recall,
        manual_recall
    ],
    "F1": [
        logistic_f1,
        manual_f1
    ],
    "Training Time (s)": [
        logistic_train_time,
        logistic_manual_train_time
    ],
    "Prediction Time (s)": [
        logistic_pred_time,
        logistic_manual_pred_time
    ]
})

print("Regression comparison")
display(regression_comparison)

print("Classification comparison")
display(classification_comparison)

Regression comparison


,Model,MAE,RMSE,R²,Training Time (s),Prediction Time (s)
0,Scikit-learn Linear Regression,0.112381,0.150048,0.152084,0.010270,0.000507
1,Manual Linear Regression,0.112381,0.150048,0.152076,0.018172,0.000113


Classification comparison


,Model,Accuracy,Precision,Recall,F1,Training Time (s),Prediction Time (s)
0,Scikit-learn Logistic Regression,0.741667,0.769953,0.926554,0.841026,0.033639,0.000641
1,Manual Logistic Regression,0.754167,0.768182,0.954802,0.851385,0.848589,0.000221


## 13. Manual Logistic Regression Optimization Experiment

The original manual implementation uses:

- learning rate = `0.01`
- epochs = `10,000`

A second experiment increases both values:

- learning rate = `0.05`
- epochs = `20,000`

This is a **hyperparameter/convergence experiment**, not an assumption that increasing these values must improve the model.

In [16]:
optimized_learning_rate = 0.05
optimized_epochs = 20000

theta_log_opt = np.zeros(X_train_log.shape[1])
n = X_train_log.shape[0]

start = time.perf_counter()

for _ in range(optimized_epochs):
    z = X_train_log @ theta_log_opt
    y_prob = sigmoid(z)

    gradient = (
        X_train_log.T @ (y_prob - y_train_log)
    ) / n

    theta_log_opt -= optimized_learning_rate * gradient

logistic_opt_train_time = time.perf_counter() - start

start = time.perf_counter()

y_prob_opt = logistic_predict_proba(
    X_test_log,
    theta_log_opt
)

y_pred_opt = (
    y_prob_opt >= 0.5
).astype(int)

logistic_opt_pred_time = time.perf_counter() - start

(
    accuracy_opt,
    precision_opt,
    recall_opt,
    f1_opt
) = classification_metrics(
    y_test_log,
    y_pred_opt
)

print("Optimized Manual Logistic Regression")
print("Accuracy        :", accuracy_opt)
print("Precision       :", precision_opt)
print("Recall          :", recall_opt)
print("F1              :", f1_opt)
print("Training time   :", logistic_opt_train_time)
print("Prediction time :", logistic_opt_pred_time)

Optimized Manual Logistic Regression
Accuracy        : 0.7333333333333333
Precision       : 0.7729468599033816
Recall          : 0.903954802259887
F1              : 0.8333333333333334
Training time   : 1.6990012000023853
Prediction time : 0.00024619999749120325


## 14. Discussion

### Predictive performance

The manual Linear Regression implementation should produce results very close to the scikit-learn Linear Regression model because both solve the same least-squares problem after using the same preprocessing and train-test split.

The Logistic Regression results can differ because scikit-learn and the manual implementation use different optimization procedures and convergence settings.

### Execution time

The manual Logistic Regression implementation performs many gradient-descent iterations in Python. Although the matrix operations inside each iteration are vectorized with NumPy, the repeated optimization loop makes training slower than scikit-learn's optimized implementation.

Prediction is a simple matrix operation for both approaches, so prediction times are much smaller than training times.

### Optimization experiment

For the tested configuration (`learning_rate = 0.05`, `epochs = 20,000`), the optimized experiment does not necessarily improve the evaluation metrics. A higher learning rate and more iterations increase computation, but they do not guarantee better generalization on the test set.

This experiment therefore demonstrates why learning rate and convergence settings need to be tuned rather than simply increased.

## 15. Final Conclusion

The assignment demonstrates the same machine-learning workflow using two approaches:

- **Scikit-learn** provides ready-to-use preprocessing, model-training, and evaluation utilities.
- **NumPy/Pandas** can reproduce the essential workflow manually.

The manual implementation includes:

- missing-value handling,
- categorical encoding,
- feature scaling,
- Linear Regression using the pseudo-inverse,
- Logistic Regression using sigmoid + gradient descent,
- vectorized NumPy computations,
- and manual evaluation metrics.

Using the same train-test samples and preprocessing logic makes the comparison between the two approaches meaningful.